# chamber_analysis – Package Demo

End-to-end walkthrough: load CSV files → metadata extraction → pyFlux (LM + HM) → interactive plots → export.

| Step | What happens |
|------|--------------|
| 1 | Configure chamber physical parameters |
| 2 | Load all CSV files from a directory |
| 3 | Inspect raw data and auxiliary metadata |
| 4 | Run the full pyFlux pipeline (all sensors × all files) |
| 5 | Visualize: time series, regression, flux, comparison |
| 6 | Export tidy summary CSV |

**Run this notebook from the directory that contains the CSV files**, or set `DATA_DIR` below.

---

In [ ]:
import pandas as pd
import datetime

import chamber_analysis as ca
from chamber_analysis import plots
from chamber_analysis.pyflux import (
    analyze_directory,
    analyze_file,
    analyze_sensor,
    GAS_META,
)

## Step 1 – Configure chamber parameters

Set the physical dimensions of your chamber. Temperature and pressure can be fixed values or `"auto"` to read mean values directly from the data columns `Temperature intern (degC)` and `Pressure intern (mbar)`.

In [ ]:
DATA_DIR = "."   # directory containing the CSV files

params = ca.ChamberParams(
    volume_L      = 13.57,    # chamber headspace volume [L]
    area_m2       = 0.04524,  # soil surface area [m²]
    temperature_C = "auto",   # read from data; or set a fixed value, e.g. 20.0
    pressure_mbar = "auto",   # read from data; or set a fixed value, e.g. 1013.25
    window_frac   = 0.8,      # fraction of measurement used for regression window
)
print(params)

## Step 2 – Load data

`load_directory` finds all `.csv` files recursively, parses the semicolon-delimited format, and returns a `dict[filename → DataFrame]`. Each DataFrame contains the raw sensor columns plus derived `time_sec` / `time_min` columns.

In [ ]:
all_dfs = ca.load_directory(DATA_DIR)

## Step 3 – Inspect raw data

### 3a – Available columns

In [ ]:
first_fname = list(all_dfs.keys())[0]
df0 = all_dfs[first_fname]

print(f"File : {first_fname}")
print(f"Rows : {len(df0)}")
print(f"Duration: {df0['time_min'].max():.1f} min")
print(f"\nColumns ({len(df0.columns)}):")
for col in df0.columns:
    print(f"  {col}")

### 3b – Auxiliary metadata

`extract_metadata` aggregates all non-gas columns into a structured `MeasurementMetadata` object. GPS uses the median; all other fields use the mean (zeros excluded — they indicate sensor not connected).

In [ ]:
meta = ca.extract_metadata(df0, first_fname)

print(f"Start    : {meta.timestamp_start}")
print(f"End      : {meta.timestamp_end}")
print(f"Duration : {meta.duration_s:.0f} s  ({meta.duration_s/60:.1f} min)")
print(f"Samples  : {meta.n_samples}")
print(f"\nGPS      : {meta.gps_latitude:.5f}°N  {meta.gps_longitude:.5f}°E  {meta.gps_altitude_m:.1f} m")
print(f"T intern : {meta.temperature_intern_C:.2f} °C")
print(f"P intern : {meta.pressure_intern_mbar:.2f} mbar")
print(f"T extern : {meta.temperature_extern_C:.2f} °C")
print(f"Humidity : {meta.humidity_intern_pct:.1f} %relH")
print(f"Battery  : {meta.battery_level_pct:.0f} %")
if meta.soil_water_content_pct is not None:
    print(f"SWC      : {meta.soil_water_content_pct:.2f} vol%")

### 3c – Time-series plot (one file)

In [ ]:
gas_sensors = list(GAS_META.keys())

fig = plots.time_series_fig(
    df0,
    sensors=gas_sensors,
    title=first_fname,
)
fig.show()

## Step 4 – Run the pyFlux pipeline

`analyze_directory` processes every file × every active sensor:

1. Resolve T and P (from data or fixed params)
2. Convert native units → ppm
3. Sliding-window R² maximisation → best linear segment
4. Fit **LM** (linear) and **HM** (Hutchinson-Mosier, non-linear) models
5. Score both models on MAE / RMSE / AICc / SE → select winner
6. Assign quality flag: `good` / `below_mdf` / `below_detection`
7. Scale flux to reporting unit (µmol or nmol m⁻² s⁻¹)

In [ ]:
results, summary = analyze_directory(all_dfs, params)

print(f"{len(results)} results  ({len(all_dfs)} files × {len(gas_sensors)} sensors)")

### 4a – Summary DataFrame

One row per file × sensor. Contains flux, statistics, environmental conditions, and all auxiliary metadata.

In [ ]:
# Key result columns
summary[[
    "filename", "sensor", "model", "flux", "flux_se", "unit",
    "r_squared", "p_value", "quality", "g_factor",
    "temperature_C", "pressure_mbar",
]].head(12)

In [ ]:
# Metadata columns included in every row
summary[[
    "filename", "sensor",
    "timestamp_start", "duration_s", "n_samples",
    "gps_latitude", "gps_longitude", "gps_altitude_m",
    "soil_water_content_pct", "soil_temperature_C",
    "battery_level_pct",
]].head(6)

### 4b – Model selection statistics

In [ ]:
print("Model choice (counts):")
print(summary.groupby(["sensor", "model"]).size().unstack(fill_value=0).to_string())

print("\nQuality flags (counts):")
print(summary.groupby(["sensor", "quality"]).size().unstack(fill_value=0).to_string())

### 4c – Accessing individual result fields

Each entry in `results` is a `SensorFluxResult` with nested `FluxResult`, `LMResult`, and `HMResult` objects.

In [ ]:
r = next(x for x in results if x.sensor == "CO2 Dynament (ppm)")

print(f"File      : {r.filename}")
print(f"Sensor    : {r.sensor}")
print(f"Flux      : {r.flux_reported:.5f} {r.report_unit}")
print(f"Model     : {r.flux_result.model}")
print(f"Quality   : {r.flux_result.quality}")
print(f"g-factor  : {r.flux_result.g_factor:.4f}  (|HM/LM| flux ratio)")
print()
print(f"── LM ──────────────────────────")
print(f"  slope    : {r.flux_result.lm.slope:.6e} ppm s⁻¹")
print(f"  R²       : {r.flux_result.lm.r_squared:.4f}")
print(f"  p-value  : {r.flux_result.lm.p_value:.2e}")
print(f"  AICc     : {r.flux_result.lm.aicc:.2f}")
print(f"  MDF      : {r.flux_result.lm.mdf:.5f} µmol m⁻² s⁻¹")
if r.flux_result.hm and r.flux_result.hm.converged:
    hm = r.flux_result.hm
    print(f"\n── HM ──────────────────────────")
    print(f"  C₀       : {hm.C0:.2f} ppm")
    print(f"  C∞       : {hm.Cinf:.2f} ppm")
    print(f"  κ        : {hm.kappa:.5f} s⁻¹  (half-life: {0.693/hm.kappa:.0f} s)")
    print(f"  R²       : {hm.r_squared:.4f}")
    print(f"  AICc     : {hm.aicc:.2f}")

## Step 5 – Visualize results

### 5a – Regression plot (one file, one sensor)

Shows the full time series, the analysis window (gold background), and both LM (red dashed) and HM (orange dotted) fits.

In [ ]:
r_co2 = next(x for x in results if x.sensor == "CO2 Dynament (ppm)")
fig = plots.regression_fig(r_co2, all_dfs[r_co2.filename])
fig.show()

### 5b – Flux bar charts per sensor

Green = LM model chosen, orange = HM chosen. Error bars = standard error (delta method). Quality symbol: ✓ good · ~ below MDF · ! below detection limit.

In [ ]:
for sensor in gas_sensors:
    fig = plots.flux_bar_fig(results, sensor)
    fig.show()

### 5c – Overview: all measurements overlaid per sensor

Faded lines = full time series. Dashed lines = regression fit in the selected window. Use the legend to toggle individual measurements on/off.

In [ ]:
fig = plots.overview_fig(results, all_dfs)
fig.show()

### 5d – LM vs HM model comparison

Left: grouped bar chart (LM vs HM flux side-by-side per file). Right: 1:1 scatter — points on the line = perfect agreement; deviation indicates non-linearity where HM and LM diverge.

In [ ]:
fig = plots.model_comparison_fig(results, "CO2 Dynament (ppm)")
fig.show()

### 5e – Battery level

In [ ]:
fig = plots.battery_fig(all_dfs)
fig.show()

## Step 6 – Export results

The summary DataFrame already contains flux, statistics, and all auxiliary metadata in a single tidy table — one row per measurement × sensor.

In [ ]:
ts  = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
out = f"Flux_pyFlux_{ts}.csv"

summary.to_csv(out, index=False, sep=";", decimal=",", encoding="utf-8-sig")
print(f"Saved: {out}  ({len(summary)} rows × {len(summary.columns)} columns)")
print(f"\nColumns: {list(summary.columns)}")

---

## Sensor configuration

To restrict analysis to specific sensors, pass an explicit list:

In [ ]:
# Analyse CO2 Dynament only
results_co2, summary_co2 = analyze_directory(
    all_dfs, params,
    sensors=["CO2 Dynament (ppm)"],
)
print(summary_co2[["filename", "sensor", "model", "flux", "unit", "quality"]])

## Adding a new sensor

Register any new sensor column in `GAS_META` before calling `analyze_directory`:

In [ ]:
from chamber_analysis.pyflux.constants import GAS_META

GAS_META["CO2 MyNewSensor (ppm)"] = {
    "factor_to_ppm": 1.0,
    "precision":     5.0,
    "report_factor": 1.0,
    "report_unit":   "µmol m⁻² s⁻¹",
}

print("Registered sensors:")
for name, meta in GAS_META.items():
    print(f"  {name:40s} → {meta['report_unit']}")